# Klasyfikacja zgłoszeń helpdesk — baseline regułowy

**Zakres:** ręczna klasyfikacja zgłoszeń, analiza danych treningowych oraz implementacja prostego klasyfikatora regułowego. Do predykcji wykorzystuję tylko tytuł i opis zgłoszenia.

## 1. Ręczna klasyfikacja

| Zgłoszenie | Kategoria | Uzasadnienie |
|---|---|---|
| A. VPN disconnects | `network` | Zgłoszenie dotyczy cyklicznego rozłączania połączenia VPN. |
| B. Suspicious email | `security` | Wiadomość z prośbą o potwierdzenie hasła może być próbą phishingu. |
| C. Windows Hello PIN | `account-access` | Niedziałający PIN uniemożliwia zalogowanie się do konta. |

**1.1–1.2.** Przy przypisywaniu kategorii istotne były zwroty: „VPN disconnects”, „confirm my password” w podejrzanej wiadomości oraz „PIN stopped working”.

**1.3.** Słowo „password” może dotyczyć zarówno resetowania hasła (`account-access`), jak i próby wyłudzenia danych (`security`). Samo słowo kluczowe nie zawsze wystarcza do podjęcia decyzji.

## 2. Zadanie i ryzyko błędu

**2.1.** Najbardziej kosztowny błąd to przypisanie próby phishingu do zwykłego problemu z dostępem, ponieważ może opóźnić reakcję na zagrożenie bezpieczeństwa.

**2.2.** Klasyfikator powinien umożliwiać skierowanie zgłoszenia do człowieka, gdy brak wyraźnego dopasowania. Ogranicza to ryzyko błędnego przydzielenia sprawy.

**2.3.** Dopuszczalne cechy to `title` i `description`. `ticket_id` jest identyfikatorem, a `expected_category` to etykieta, której nie wolno używać jako wejścia klasyfikatora.

## 3. Podział danych

**3.1.** Zbiór otwierany po zamrożeniu rozwiązania: **test**. Zbiór do projektowania reguł: **train**. Zbiór do wyboru wersji: **validation**.

**3.2.** Poprawianie reguł po poznaniu wyników na teście prowadziłoby do dopasowania do testu, który przestałby być niezależną oceną.

## 4. Dane treningowe i walidacja

In [1]:
from pathlib import Path
import pandas as pd

ALLOWED_CATEGORIES = [
    "account-access", "network", "hardware",
    "software", "security", "other",
]
REQUIRED_COLUMNS = {
    "ticket_id", "title", "description",
    "expected_category", "priority", "language",
}

print("Konfiguracja: OK")


Konfiguracja: OK


In [2]:
DATA_PATH = Path("tickets_train.csv")
assert DATA_PATH.exists(), (
    f"Nie znaleziono pliku {DATA_PATH.name}. "
    f"Umieść go obok notebooka. Bieżący katalog: {Path.cwd()}"
)

tickets = pd.read_csv(DATA_PATH)
print("Wymiary:", tickets.shape)
print("Kolumny:", tickets.columns.tolist())
display(tickets.head(3))


Wymiary: (200, 6)
Kolumny: ['ticket_id', 'title', 'description', 'expected_category', 'priority', 'language']


,ticket_id,title,description,expected_category,priority,language
0,TCK-0001,VPN disconnects,My VPN disconnects every few minutes when I wo...,network,normal,en
1,TCK-0002,Suspicious email,I received an email asking me to confirm my pa...,security,high,en
2,TCK-0003,Slow downloads over VPN,"When connected to VPN, downloads from SharePoi...",network,normal,en


In [3]:
def validate_dataset(data: pd.DataFrame) -> None:
    missing_columns = REQUIRED_COLUMNS - set(data.columns)
    assert not missing_columns, f"Brak kolumn: {sorted(missing_columns)}"
    assert data.shape == (200, 6), f"Oczekiwano (200, 6), otrzymano {data.shape}"

    assert data["ticket_id"].notna().all(), "Brak ticket_id"
    assert data["ticket_id"].is_unique, "ticket_id nie są unikatowe"

    for column in ["title", "description", "expected_category", "priority", "language"]:
        assert data[column].notna().all(), f"Braki w kolumnie {column}"

    unknown = set(data["expected_category"]) - set(ALLOWED_CATEGORIES)
    assert not unknown, f"Nieznane kategorie: {sorted(unknown)}"

validate_dataset(tickets)
print("Walidacja danych: OK")


Walidacja danych: OK


### 4.5. Podstawowe informacje o zbiorze

Zbiór zawiera **200 zgłoszeń i 6 kolumn**. Najliczniejszą kategorią jest **software (50)**, a najrzadsze są **security (20)** i **other (20)**.

In [4]:
category_counts = tickets["expected_category"].value_counts()
print("Rozmiar zbioru:", tickets.shape)
print("Braki danych w całym zbiorze:", int(tickets.isna().sum().sum()))
print("Duplikaty całych wierszy:", int(tickets.duplicated().sum()))
display(category_counts.rename("liczba_zgloszen"))


Rozmiar zbioru: (200, 6)
Braki danych w całym zbiorze: 0
Duplikaty całych wierszy: 0


expected_category
software          50
account-access    44
network           36
hardware          30
security          20
other             20
Name: liczba_zgloszen, dtype: int64

## 5. Przegląd przykładów i sygnały tekstowe

In [5]:
display(
    tickets[["title", "description", "expected_category"]]
    .head(20)
    .style.hide(axis="index")
)


title,description,expected_category
VPN disconnects,My VPN disconnects every few minutes when I work from home.,network
Suspicious email,I received an email asking me to confirm my password.,security
Slow downloads over VPN,"When connected to VPN, downloads from SharePoint are about 100 KB/s.",network
Nie działa podgląd PDF,"W Outlooku nie mogę podejrzeć załączników PDF, pojawia się komunikat o braku programu do podglądu.",software
Excel freezes with large files,Excel stops responding whenever I open the monthly sales report (about 80 MB). Smaller files work fine.,software
Bateria szybko się rozładowuje,"Bateria w laptopie trzyma tylko godzinę, wcześniej wytrzymywała cały dzień.",hardware
Wi-Fi lagguje na callach,"Na Wi-Fi w open space wszystko lagguje, zwłaszcza calle na Teamsach. Na kablu jest ok.",network
PowerPoint template missing,The new corporate PowerPoint template is not available under File > New. Colleagues have it.,software
Windows Hello PIN not available,My Windows Hello PIN stopped working and it says 'Something happened and your PIN isn't available'.,account-access
CRM app shows error 500,The Salesforce integration in our CRM client throws 'Internal Server Error 500' when I save a contact.,software


| Kategoria | Silne sygnały | Możliwy konflikt |
|---|---|---|
| `security` | `phishing`, `suspicious email`, `trojan`, `antivirus alert` | „password” może oznaczać problem z logowaniem. |
| `account-access` | `permission`, `locked account`, `pin`, `password reset` | „pin” może wystąpić w kontekście innej usterki. |
| `network` | `vpn`, `wi-fi`, `internet`, `dns` | Błąd uwierzytelniania do VPN może dotyczyć konta. |
| `hardware` | `battery`, `charger`, `printer`, `keyboard` | Problem ze sterownikiem drukarki może dotyczyć oprogramowania. |
| `software` | `error`, `install`, `outlook`, `excel` | Podejrzana wiadomość w Outlooku to kwestia bezpieczeństwa. |
| `other` | Brak dopasowania do konkretnych kategorii | Brak dopasowania może wynikać z niedostatecznych reguł. |

Sygnały zostały dobrane dla pięciu konkretnych kategorii. `other` stanowi kategorię domyślną i wymaga ręcznej weryfikacji.

## 6. Klasyfikator regułowy

Klasyfikacja polega na wyszukiwaniu fraz w połączonym tytule i opisie zgłoszenia. Przy dopasowaniu do kilku kategorii rozstrzyga kolejność reguł; `security` jest sprawdzane jako pierwsze.

In [6]:
RULES = {
    "security": ["phishing", "suspicious email", "trojan", "antivirus alert"],
    "account-access": ["permission", "locked account", "pin", "password reset"],
    "network": ["vpn", "wi-fi", "internet", "dns"],
    "hardware": ["battery", "charger", "printer", "keyboard"],
    "software": ["error", "install", "outlook", "excel"],
}

number_of_terms = sum(len(terms) for terms in RULES.values())
print("Liczba sygnałów przed poprawką:", number_of_terms)
assert number_of_terms >= 10


Liczba sygnałów przed poprawką: 20


In [7]:
def normalize_text(text) -> str:
    """Zamienia brak tekstu na pusty napis, usuwa nadmiar spacji i zmienia litery na małe."""
    if text is None:
        return ""
    return " ".join(str(text).lower().split())


def classify_ticket(title: str, description: str) -> dict:
    """Zwraca kategorię, umowną pewność i informację o potrzebie weryfikacji."""
    text = normalize_text(f"{normalize_text(title)} {normalize_text(description)}")

    for category, terms in RULES.items():
        if any(normalize_text(term) in text for term in terms):
            return {
                "category": category,
                "confidence": 0.7,
                "needs_human_review": False,
            }

    return {
        "category": "other",
        "confidence": 0.0,
        "needs_human_review": True,
    }

print("Funkcja klasyfikująca: gotowa")


Funkcja klasyfikująca: gotowa


### 6.1. Sprawdzenie przykładowych zgłoszeń

In [8]:
opening_cases = [
    ("VPN disconnects", "My VPN disconnects every few minutes when I work from home."),
    ("Suspicious email", "I received an email asking me to confirm my password."),
    ("Windows Hello PIN", "My PIN stopped working and Windows says it is not available."),
]

for title, description in opening_cases:
    print(title, "->", classify_ticket(title, description))


VPN disconnects -> {'category': 'network', 'confidence': 0.7, 'needs_human_review': False}
Suspicious email -> {'category': 'security', 'confidence': 0.7, 'needs_human_review': False}
Windows Hello PIN -> {'category': 'account-access', 'confidence': 0.7, 'needs_human_review': False}


## 7. Testy kontraktu i ocena na train

In [9]:
def run_contract_tests(classifier) -> None:
    cases = [
        ("", ""),
        ("Example", "Simple description"),
        ("ZAŻÓŁĆ", "mixed CASE 123"),
        (None, None),
    ]

    for title, description in cases:
        result = classifier(title, description)
        assert isinstance(result, dict)
        assert set(result) == {"category", "confidence", "needs_human_review"}
        assert result["category"] in ALLOWED_CATEGORIES
        assert isinstance(result["confidence"], (int, float))
        assert 0.0 <= result["confidence"] <= 1.0
        assert isinstance(result["needs_human_review"], bool)

    print("Testy kontraktu: OK")

run_contract_tests(classify_ticket)


Testy kontraktu: OK


In [10]:
def evaluate(data: pd.DataFrame, classifier) -> pd.DataFrame:
    outputs = data.apply(
        lambda row: classifier(row["title"], row["description"]),
        axis=1,
    )

    result = data.copy()
    result["predicted_category"] = [item["category"] for item in outputs]
    result["confidence"] = [item["confidence"] for item in outputs]
    result["needs_human_review"] = [item["needs_human_review"] for item in outputs]
    return result

results = evaluate(tickets, classify_ticket)
display(results.head(5))


,ticket_id,title,description,expected_category,priority,language,predicted_category,confidence,needs_human_review
0,TCK-0001,VPN disconnects,My VPN disconnects every few minutes when I wo...,network,normal,en,network,0.7,False
1,TCK-0002,Suspicious email,I received an email asking me to confirm my pa...,security,high,en,security,0.7,False
2,TCK-0003,Slow downloads over VPN,"When connected to VPN, downloads from SharePoi...",network,normal,en,network,0.7,False
3,TCK-0004,Nie działa podgląd PDF,"W Outlooku nie mogę podejrzeć załączników PDF,...",software,low,pl,software,0.7,False
4,TCK-0005,Excel freezes with large files,Excel stops responding whenever I open the mon...,software,normal,en,software,0.7,False


In [11]:
results["is_correct"] = (
    results["expected_category"] == results["predicted_category"]
)
train_accuracy = results["is_correct"].mean()

print(f"Accuracy na train: {train_accuracy:.3f} ({train_accuracy:.1%})")
print("To nie jest jeszcze pomiar generalizacji.")


Accuracy na train: 0.375 (37.5%)
To nie jest jeszcze pomiar generalizacji.


### 7.1. Analiza błędów przed poprawką

| ID | Oczekiwana kategoria | Predykcja | Przyczyna błędu | Proponowana poprawka |
|---|---|---|---|---|
| `TCK-0006` | `hardware` | `other` | Reguły rozpoznają angielskie „battery”, ale nie polskie „bateria”. | Dodać „bateria” do `hardware`. |
| `TCK-0008` | `software` | `other` | Brak frazy „powerpoint” w sygnałach kategorii. | Rozważyć dodanie „powerpoint” w kolejnej wersji. |

W tej wersji wprowadzam **jedną poprawkę**: dodaję „bateria” do kategorii `hardware`.

In [12]:
error_columns = [
    "ticket_id", "title", "description",
    "expected_category", "predicted_category",
    "confidence", "needs_human_review",
]
errors = results.loc[~results["is_correct"], error_columns]

print("Liczba błędów:", len(errors))
display(errors.head(10).style.hide(axis="index"))


Liczba błędów: 125


ticket_id,title,description,expected_category,predicted_category,confidence,needs_human_review
TCK-0006,Bateria szybko się rozładowuje,"Bateria w laptopie trzyma tylko godzinę, wcześniej wytrzymywała cały dzień.",hardware,other,0.000000,True
TCK-0008,PowerPoint template missing,The new corporate PowerPoint template is not available under File > New. Colleagues have it.,software,other,0.000000,True
TCK-0013,Teams screen sharing not working,"When I try to share my screen in Teams, the other participants only see a black window.",software,other,0.000000,True
TCK-0016,Salary file sent to wrong recipient,I accidentally sent a file with employee salaries to an external email address.,security,other,0.000000,True
TCK-0017,Password posted in Teams chat,I pasted my password into a Teams chat with the whole team by mistake.,security,other,0.000000,True
TCK-0018,Zoom audio not working,"Zoom doesn't detect any microphone, although the same microphone works in Teams on this laptop.",software,other,0.000000,True
TCK-0020,Prośba o instalację programu,"Proszę o zainstalowanie programu 7-Zip na moim komputerze, nie mam uprawnień administratora.",software,other,0.000000,True
TCK-0022,Prośba o aktualizację przeglądarki,Firefox na moim komputerze jest bardzo stary i część stron się nie wyświetla. Proszę o aktualizację.,software,other,0.000000,True
TCK-0023,Adobe Acrobat license expired,Acrobat says my trial has expired and I cannot edit PDFs anymore. I thought we had a company license.,software,other,0.000000,True
TCK-0024,New phone for Authenticator,I got a new phone and need to move Microsoft Authenticator. The old phone was already reset.,account-access,other,0.000000,True


### 7.2. Jedna poprawka i ponowna weryfikacja

In [13]:
# Punkt 7.5: dokładnie jedna zmiana w regułach
RULES["hardware"].append("bateria")

# Punkt 7.6: uruchamiam te SAME testy kontraktu (nie zmieniam ich)
run_contract_tests(classify_ticket)

# Ponowny pomiar po zmianie — nadpisuję wynik, aby podsumowanie było aktualne
accuracy_before = float(train_accuracy)
errors_before = int((~results["is_correct"]).sum())
results = evaluate(tickets, classify_ticket)
results["is_correct"] = results["expected_category"] == results["predicted_category"]
train_accuracy = results["is_correct"].mean()

print(f"Przed poprawką: {accuracy_before:.1%}, liczba błędów: {errors_before}")
print(f"Po poprawce:   {train_accuracy:.1%}, liczba błędów: {(~results['is_correct']).sum()}")
print("TCK-0006 po poprawce:", results.loc[results['ticket_id'] == 'TCK-0006', 'predicted_category'].iloc[0])


Testy kontraktu: OK
Przed poprawką: 37.5%, liczba błędów: 125
Po poprawce:   38.0%, liczba błędów: 124
TCK-0006 po poprawce: hardware


In [14]:
# Punkt 7.7: zapisz wynik baseline v0
baseline_summary = {
    "version": "baseline-v0",
    "dataset": "train",
    "accuracy_before": round(float(accuracy_before), 3),
    "accuracy_after": round(float(train_accuracy), 3),
    "number_of_terms": sum(len(terms) for terms in RULES.values()),
    "analysed_errors": ["TCK-0006", "TCK-0008"],
    "single_change": "Dodano 'bateria' do hardware (TCK-0006).",
    "contract_tests": "OK (przed zmianą i po zmianie)",
}
baseline_summary

{'version': 'baseline-v0',
 'dataset': 'train',
 'accuracy_before': 0.375,
 'accuracy_after': 0.38,
 'number_of_terms': 21,
 'analysed_errors': ['TCK-0006', 'TCK-0008'],
 'single_change': "Dodano 'bateria' do hardware (TCK-0006).",
 'contract_tests': 'OK (przed zmianą i po zmianie)'}

## 8. Exit ticket

**1. Dlaczego `expected_category` nie może być cechą?**  
To etykieta, którą klasyfikator ma przewidzieć. Jej wykorzystanie w danych wejściowych oznaczałoby wyciek poprawnej odpowiedzi.

**2. Czym różni się test kontraktu od pomiaru jakości?**  
Test kontraktu sprawdza, czy funkcja zwraca właściwą strukturę i typy danych. Pomiar jakości ocenia zgodność predykcji z poprawnymi kategoriami.

**3. Przykład reguły i sytuacji, w której może zawieść.**  
Reguła `vpn` → `network` może pomylić się przy zgłoszeniu „VPN authentication failed”, gdy rzeczywistą przyczyną jest zablokowane konto.

**4. Dlaczego wynik na train nie wystarcza do oceny systemu?**  
Reguły opracowano na podstawie danych treningowych, dlatego wynik na `train` nie potwierdza skuteczności dla nowych zgłoszeń. Do niezależnej oceny potrzebne są zbiory `validation` i `test`.

**Wniosek:** baseline jest prosty i zrozumiały, ale ma ograniczoną skuteczność, szczególnie przy zróżnicowanym słownictwie. Wynik na `train` jest wyłącznie pomiarem roboczym.